# 7.4 Your turn: our relationship as a map of places

> **Question:** what does our relationship look like from a geographical point of view,
> going by our WhatsApp chat?

**My expectation:** during the long distance we talk about many cities, in France and in
the Netherlands. When I am pregnant, the cities stop coming up.

The method, from 7.1–7.3:

1. **Nodes are places**: every city (and Corsica) that came up in the chat, from a list I
   made by hand (`settings.PLACES`), coloured by country: France, the Netherlands, elsewhere.
2. **The edge rule, as a sentence:** *two places are connected when both came up on the
   same day.* That is 07.1's "who was nearby", with a whole day as the window, on places
   instead of people.
3. **Everything is counted in days**, not messages: ten messages about one train to
   Marseille are one day of Marseille. A dot grows with the days its place came up; a line
   gets heavier with the days the two places shared.
4. **In 7.4.4 and 7.4.7 the x-axis is time.** A place sits at the *median* day it came up,
   so its position says when we typically talked about it. The y-axis only separates the
   countries.
5. **From 7.4.5 on there is no time axis** (feedback, 2026-11-03: a network and a time axis
   don't go together; "was I only in Paris in March 2024?"). The places sit on a circle, one
   arc per country; the position says the country and nothing else. Under every picture a
   short text says what a dot, a line and its width mean.

In [ ]:
import networkx as nx
import pandas as pd
from goad_toolkit.datatransforms import Pipeline
from goad_toolkit.visualizer import LinePlot, PlotSettings

from roos_pipeline import settings
from roos_pipeline.places import (
    GraphPlot,
    TimelineGraphPlot,
    UnmergeMessages,
    author_shares,
    circle_layout,
    lane_layout,
    place_days,
    place_graph,
    prune,
    who_names,
)
from roos_pipeline.people import (
    MatrixPlot,
    benji_graph,
    load_people,
    mentions,
    people_graph,
    person_sides,
    side_order,
    star_layout,
)
from roos_pipeline.prepare import CanonicalAuthor, load_author_aliases
from roos_pipeline.settings import Paths

PREGNANCY = (settings.PREGNANCY_START, settings.BIRTH_DATE, "pregnancy")
MARGIN = pd.Timedelta(days=15)  # a little room left and right of the period

## 7.4.1 Messages back apart

From `1_messages.parquet` (the pipeline's stage 1: one author name per person, local time).
After an "afbeelding weggelaten" line the export sometimes glues the next messages into the
same row, each behind its own `[13-08-2023, 14:37:06] Roos:` header, sometimes a header from
another day. In lesson 6 `StripQuotedReply` cut those off. Here `UnmergeMessages` splits them
back into their own rows, with their own time, because a place in them came up on *that* day.

Those headers carry the display name of *that* moment, so Louis's old names come back
("Louis Ancien", ...). `CanonicalAuthor` merges them again, with the same aliases as stage 1;
that only matters for 7.4.9, where it counts who named a place.

In [ ]:
raw = pd.read_parquet(Paths().messages)
messages = (
    Pipeline()
    .add(UnmergeMessages)
    .add(CanonicalAuthor, aliases=load_author_aliases(Paths().config))
    .apply(raw.copy())
)
messages = messages[messages["message"] != ""]
print(f"{len(raw):,} rows in the export -> {len(messages):,} messages after splitting, "
      f"{messages['ts_local'].min():%d %b %Y} to {messages['ts_local'].max():%d %b %Y}")
print("authors:", ", ".join(f"{a} {n:,}" for a, n in messages["author"].value_counts().items()))

## 7.4.2 Which places came up

`place_days` checks every message against every place in `settings.PLACES` and keeps one row
per (day, place). The list is hand-made: I went through all capitalised words in the chat and
checked the ambiguous ones. Left out on purpose, so they count as *nothing*, not as a place:
"Nice" (almost always the English word), "Tours" ("deux tours"), "Spa", "Houten" ("houten
deur"), Luxembourg (the country), Schiphol (an airport) and neighbourhoods like Montmartre or
Overvecht, which belong to a city already on the list.


In [ ]:
days = place_days(messages)
per_place = days.groupby(["country", "place"]).size().rename("days").reset_index()
print(f"{days['place'].nunique()} places came up, on {days['day'].nunique()} different days\n")
for country in settings.COUNTRY_ORDER:
    top = per_place[per_place["country"] == country].nlargest(8, "days")
    print(f"{country:12s}", ", ".join(f"{p} {d}" for p, d in zip(top["place"], top["days"])))

## 7.4.3 The graph of the long-distance period

The long-distance period: from our first message (12 Aug 2023) to Louis's move to Utrecht
(`settings.MOVE_DATE`, 1 Nov 2024). The grey area in the plots is the pregnancy, from
`settings.PREGNANCY_START` (due-date convention, 41 weeks before Benji) to his birth. So in
this period, only its first three months are visible.

Before the picture, the numbers (07.1.8: *a network diagram is very good at looking like a
finding*).


In [ ]:
long_distance = place_graph(days, settings.LONG_DISTANCE_START, settings.MOVE_DATE)

strongest = sorted(long_distance.edges(data="weight"), key=lambda e: -e[2])[:8]
once = sum(1 for _, _, w in long_distance.edges(data="weight") if w == 1)
print(f"{long_distance.number_of_nodes()} places, {long_distance.number_of_edges()} links, "
      f"density {nx.density(long_distance):.2f}")
print(f"links that rest on a single day: {once} of {long_distance.number_of_edges()} "
      f"({once / long_distance.number_of_edges():.0%})\n")
print("the strongest links (days together):")
for u, v, w in strongest:
    print(f"  {u} - {v}: {w}")

## 7.4.4 Visualisation 1: every place

Every place that came up in the long-distance period, every link. Names only for places of
at least 10 days; the smaller dots are still there.


In [ ]:
ld_xlim = (settings.LONG_DISTANCE_START - MARGIN, settings.MOVE_DATE + MARGIN)

all_places = PlotSettings(
    figsize=(14, 6.5),
    title="Every place that came up on the same day, Aug 2023 - Nov 2024 (long distance)",
)
fig, ax = TimelineGraphPlot(all_places).plot(
    graph=long_distance, pos=lane_layout(long_distance), shade=PREGNANCY, label_min=10, xlim=ld_xlim
)

## 7.4.5 Visualisation 2: clutter reduced, without a time axis

My rules (`settings.MIN_PLACE_DAYS`, `MIN_LINK_DAYS`, `MAX_LINKS`, `ALWAYS_KEEP`), applied by
`prune` in this order:

1. **Places:** a place stays if it came up on at least 10 days. Exception: Corsica, the island
   where we met, always stays.
2. **Weak links:** a link stays if the two places came up together on at least 5 days.
3. **The 3 strongest links per place:** a link stays if it is in the top 3 of *at least one*
   of its places. So every small place keeps its link to its hub, and a hub like Utrecht can
   end up with more than 3.
4. **Nobody alone:** a place that has no link left gets back its single strongest one, even
   below 5 days. That is how Corsica (4 days with Paris) stays connected.

When two links are equally strong, the one to the place that came up on more days wins. The
order matters: step 3 ranks only the links left after steps 1 and 2.

The picture has no time axis (see point 5 at the top): the Netherlands run down the right of
the circle, France comes back up the left, elsewhere sits at the bottom, biggest place first.
`how_to_read` writes the text under the picture, with the real numbers of that graph.

In [ ]:
steps = {
    "start": long_distance,
    f"1. places >= {settings.MIN_PLACE_DAYS} days (+ Corsica)": prune(long_distance, min_link=0, max_links=10_000, no_loners=False),
    f"2. links >= {settings.MIN_LINK_DAYS} days": prune(long_distance, max_links=10_000, no_loners=False),
    f"3. top {settings.MAX_LINKS} of one of its places": prune(long_distance, no_loners=False),
    "4. nobody alone": prune(long_distance),
}
for label, graph in steps.items():
    alone = sum(1 for n in graph if graph.degree(n) == 0)
    print(f"{label:34s} {graph.number_of_nodes():3d} places, {graph.number_of_edges():3d} links, "
          f"{alone:2d} alone, {nx.number_connected_components(graph)} piece(s)")

reduced = steps["4. nobody alone"]
print()
for place in ["Corsica", "Terschelling"]:
    links = ", ".join(f"{other} ({reduced[place][other]['weight']} days)" for other in reduced[place])
    print(f"{place}, {reduced.nodes[place]['days']} days: linked to {links}")

In [ ]:
COUNTRY_LEGEND = {"Frankrijk": settings.COUNTRY_COLORS["France"],
                  "Nederland": settings.COUNTRY_COLORS["Netherlands"],
                  "Elders": settings.COUNTRY_COLORS["Elsewhere"]}


def country_colors(graph):
    return {n: settings.COUNTRY_COLORS[d["country"]] for n, d in graph.nodes(data=True)}


def how_to_read(graph, dots="gekleurd naar land"):
    """The text under a picture: what a dot, a line and its width mean, in this graph's numbers."""
    u, v, most = max(graph.edges(data="weight"), key=lambda e: e[2])
    least = min(w for _, _, w in graph.edges(data="weight"))
    return (
        f"Zo lees je dit. Elke stip is een plek die we in onze WhatsApp-chat noemden, {dots}; "
        f"hoe groter de stip, hoe meer dagen die plek genoemd werd. Een lijn betekent: die twee "
        f"plekken werden op dezelfde dag allebei genoemd, door wie maakt niet uit. Hoe dikker de "
        f"lijn, hoe meer dagen dat gebeurde: de dikste, {u}-{v}, staat voor {most} dagen, de "
        f"dunste voor {least}. Geteld zijn dagen, geen berichten, en een lijn zegt niet dat we "
        f"tussen die plekken reisden. Alleen plekken van minstens {settings.MIN_PLACE_DAYS} dagen "
        f"staan erin (Corsica, waar we elkaar ontmoetten, altijd), met per plek de "
        f"{settings.MAX_LINKS} sterkste lijnen. De plek op de cirkel zegt alleen het land: "
        f"rechts Nederland, links Frankrijk, onderaan elders."
    )


clean = PlotSettings(figsize=(11, 9), title="De geografische kaart tijdens onze langeafstandsrelatie")
fig, ax = GraphPlot(clean).plot(
    graph=reduced,
    pos=circle_layout(reduced),
    colors=country_colors(reduced),
    legend=COUNTRY_LEGEND,
    caption=how_to_read(reduced),
)

### Why rule 3 is the loose version

My first version was strict: a link only stayed if it was in the top 3 of *both* its places,
so no place could have more than 3. That left only the triangle Paris–Utrecht–Amsterdam plus
Rotterdam and Marseille, and most places stood alone: Utrecht's top 3 was full, so
Zoetermeer's strongest link (to Utrecht) was dropped. The price of the loose version: the
hubs now have more than 3 links.


In [ ]:
strict = prune(long_distance, both_ends=True, no_loners=False)
print(f"strict (top 3 of both places): {strict.number_of_edges():2d} links, "
      f"{sum(1 for n in strict if strict.degree(n) == 0)} places alone")
print(f"loose + nobody alone:          {reduced.number_of_edges():2d} links, "
      f"Paris {reduced.degree('Paris')}, Utrecht {reduced.degree('Utrecht')}, "
      f"Amsterdam {reduced.degree('Amsterdam')}")

## 7.4.6 Do the cities really stop during the pregnancy?

That was my expectation, and maybe the title. The network can't answer it well: a dot sits at
the *median* day of its place, so Paris and Utrecht end up in the middle of the period however
the mentions are spread. So first a check outside the network: **how many place-days per
month** (one place on one day = 1).


In [ ]:
per_month = (
    days.assign(month=days["day"].dt.to_period("M").dt.start_time)
    .groupby("month").size().rename("place_days").reset_index()
)

check = PlotSettings(
    figsize=(12, 4),
    title="Places that came up, per month",
    xlabel="",
    ylabel="place-days per month",
)
fig, ax = LinePlot(check).plot(data=per_month, x="month", y="place_days", marker="o", color=settings.GREY)
ax.axvspan(settings.PREGNANCY_START, settings.BIRTH_DATE, color=settings.LIGHT_GREY, alpha=0.45, lw=0)
ax.axvline(settings.MOVE_DATE, color=settings.DARK_GREY, linestyle="--", linewidth=1)
top = per_month["place_days"].max()
ax.annotate(" Louis moves", (settings.MOVE_DATE, top), color=settings.DARK_GREY)
ax.annotate(" pregnancy", (settings.PREGNANCY_START, top * 0.9), style="italic", color=settings.DARK_GREY)


def mean_per_month(start, end):
    window = per_month[(per_month["month"] >= start) & (per_month["month"] < end)]
    return window["place_days"].mean()


periods = {
    "long distance, before the pregnancy": (settings.LONG_DISTANCE_START, settings.PREGNANCY_START),
    "long distance, first months pregnant": (settings.PREGNANCY_START, settings.MOVE_DATE),
    "living together, rest of the pregnancy": (settings.MOVE_DATE, settings.BIRTH_DATE),
    "after Benji": (settings.BIRTH_DATE, per_month["month"].max() + pd.Timedelta(days=1)),
}
for label, (start, end) in periods.items():
    print(f"{label:40s} {mean_per_month(start, end):5.1f} place-days a month")

**What this shows** (check it against the numbers above after a re-run): in the first
three months of the pregnancy, places come up as often as before. The drop comes in
November 2024, when Louis moves to Utrecht and we no longer have to plan trains between Paris
and the Netherlands. So "the cities stop when I'm pregnant" is not what the chat says; "the
cities stop when we live together" fits better. Two things change close together, though
(pregnant in August, the move in November), so this is a description, not a cause.


## 7.4.7 Visualisation 3: long distance and the whole pregnancy

The longer period: from our first message to Benji's birth, so the grey area is complete.
Same clutter rules as visualisation 2; with even more days, the full graph only gets busier.


In [ ]:
longer = place_graph(days, settings.LONG_DISTANCE_START, settings.BIRTH_DATE)
longer_reduced = prune(longer)
print(f"all: {longer.number_of_nodes()} places, {longer.number_of_edges()} links -> "
      f"reduced: {longer_reduced.number_of_nodes()} places, {longer_reduced.number_of_edges()} links")
for name in longer_reduced:
    if name in long_distance:
        before, after = long_distance.nodes[name]["when"], longer.nodes[name]["when"]
        if abs(after - before) > pd.Timedelta(days=30):
            print(f"  {name}: median day {before:%b %Y} -> {after:%b %Y}")

longer_plot = PlotSettings(
    figsize=(15, 6.5),
    title="The strongest links between places, Aug 2023 - May 2025 (long distance and pregnancy)",
)
fig, ax = TimelineGraphPlot(longer_plot).plot(
    graph=longer_reduced,
    pos=lane_layout(longer_reduced),
    shade=PREGNANCY,
    xlim=(settings.LONG_DISTANCE_START - MARGIN, settings.BIRTH_DATE + MARGIN),
)
ax.axvline(settings.MOVE_DATE, color=settings.DARK_GREY, linestyle="--", linewidth=1, zorder=0)
_ = ax.annotate(" Louis moves", (settings.MOVE_DATE, -0.6), color=settings.DARK_GREY, fontsize=9)

## 7.4.8 Visualisation 4: our whole relationship, without a time axis

From our first message to the last one in the export: long distance, living together, the
pregnancy and Benji. Same rules and same circle as visualisation 2, so the two can be put
side by side.

In [ ]:
last_day = days["day"].max()
whole = place_graph(days, settings.LONG_DISTANCE_START, last_day + pd.Timedelta(days=1))
whole_reduced = prune(whole)
print(f"all: {whole.number_of_nodes()} places, {whole.number_of_edges()} links -> "
      f"reduced: {whole_reduced.number_of_nodes()} places, {whole_reduced.number_of_edges()} links")
print("new compared to the long distance:", ", ".join(sorted(set(whole_reduced) - set(reduced))) or "-")
print("gone compared to the long distance:", ", ".join(sorted(set(reduced) - set(whole_reduced))) or "-")

whole_plot = PlotSettings(
    figsize=(11, 9),
    title=f"Plekken die op dezelfde dag genoemd werden, onze hele relatie "
          f"({settings.LONG_DISTANCE_START:%Y} - {last_day:%Y})",
)
fig, ax = GraphPlot(whole_plot).plot(
    graph=whole_reduced,
    pos=circle_layout(whole_reduced),
    colors=country_colors(whole_reduced),
    legend=COUNTRY_LEGEND,
    caption=how_to_read(whole_reduced),
)

## 7.4.9 Visualisation 5: who names which place?

The same graph as visualisation 4, but the colour now says who names a place, not its country.
My rule (`settings.SHARED_LOW`, `SHARED_HIGH`):

- **Counted in days, per person:** on how many days did Roos name Utrecht, on how many days
  Louis? A day on which we both did counts for both of us.
- **Roos's share** = Roos's days / (Roos's days + Louis's days).
- **60% or more** is Roos's place, **40% or less** (so 60% or more for Louis) is Louis's,
  **in between** it is shared.

Before the picture, the numbers, and one to compare against: Roos writes more messages than
Louis, so what is her share over *all* places together?

In [ ]:
shares = author_shares(messages)
shares["who"] = shares["share"].map(who_names)
in_graph = shares.loc[list(whole_reduced)].sort_values("share")
for place, row in in_graph.iterrows():
    print(f"{place:13s} Roos {row['Roos']:3d}  Louis {row['Louis']:3d}  Roos {row['share']:4.0%}  -> {row['who']}")

all_days = shares[["Roos", "Louis"]].sum()
print(f"\nall places together: Roos {all_days['Roos'] / all_days.sum():.0%} of the place-days; "
      f"all messages: Roos {(messages['author'] == 'Roos').mean():.0%}")
close = in_graph[(in_graph["share"] - settings.SHARED_HIGH).abs().lt(0.03)
                 | (in_graph["share"] - settings.SHARED_LOW).abs().lt(0.03)]
print("within 3 points of a boundary:", ", ".join(close.index) or "-")

In [ ]:
WHO_COLORS = {"Roos": settings.ROOS_COLOR, "Louis": settings.LOUIS_COLOR, "shared": settings.SHARED_COLOR}
low, high = settings.SHARED_LOW, settings.SHARED_HIGH
who_legend = {
    f"Roos noemt het vaker ({high:.0%} of meer)": settings.ROOS_COLOR,
    f"gedeeld ({low:.0%} - {high:.0%})": settings.SHARED_COLOR,
    f"Louis noemt het vaker ({1 - low:.0%} of meer)": settings.LOUIS_COLOR,
}
who_plot = PlotSettings(
    figsize=(11, 9),
    title=f"Wie noemt welke plek? Onze hele relatie ({settings.LONG_DISTANCE_START:%Y} - {last_day:%Y})",
)
fig, ax = GraphPlot(who_plot).plot(
    graph=whole_reduced,
    pos=circle_layout(whole_reduced),
    colors={n: WHO_COLORS[shares.loc[n, "who"]] for n in whole_reduced},
    legend=who_legend,
    caption=how_to_read(
        whole_reduced,
        dots=("gekleurd naar wie hem vaker noemde: het aandeel van de dagen waarop Roos de plek "
              "noemde, tegenover Louis. Een dag waarop we hem allebei noemden telt voor ons allebei"),
    ),
)

## 7.4.10 Visualisation 6: who is named near Benji? Looking for babysitters

The same kind of picture, now with people instead of places, and one question: **who comes up
when we talk about Benji?** That is where to start looking for a babysitter. My rules
(2026-11-03, `settings.BENJI_*`, `MIN_NEAR_DAYS`):

1. **Nodes are people**, from a list in `config.toml` `[people]`, not in the repo: these are
   names of family and friends. It is a *candidate* list, made from the capitalised words
   near "Benji"; I still have to check it (cross out what isn't a person, add who is missing).
2. **Family words count per writer.** "Mon père" from me is my father, from Louis it is his;
   "ton père" from Louis is mine. So they become `vader Roos`, `moeder Louis`, ... "Papa" and
   "maman" are left out: near Benji they are mostly us.
3. **The edge rule, as a sentence:** *a person is linked to Benji on a day when their name
   came up within 30 minutes before or after a message naming Benji.* From the first
   "Benji" in the chat (January 2025, still pregnant) to the end of the export.
4. **Counted in days**, like the places. Only links to Benji (a star): no lines between people.
5. **Clutter:** a person needs at least `MIN_NEAR_DAYS` days near Benji (a starting point).

In [ ]:
people = load_people(Paths().config)
is_benji = messages["message"].str.contains(settings.BENJI_PATTERN, case=False, regex=True)
benji_start = messages.loc[is_benji, "ts_local"].min().normalize()
since_benji = messages[messages["ts_local"] >= benji_start]
benji_times = since_benji.loc[is_benji[since_benji.index], "ts_local"]

named = mentions(since_benji, people)
everyone = benji_graph(named, benji_times, min_near=1)
benji_star = benji_graph(named, benji_times)
print(f"from {benji_start:%d %b %Y}: Benji on {everyone.nodes[settings.BENJI]['days']} days, "
      f"{everyone.number_of_nodes() - 1} people near him at least once, "
      f"{benji_star.number_of_nodes() - 1} on {settings.MIN_NEAR_DAYS}+ days\n")
print(f"{'':14s} {'near':>5s} {'all':>4s}  share of their days near Benji")
for person in sorted(everyone[settings.BENJI], key=lambda p: -everyone[settings.BENJI][p]["weight"]):
    near_days, all_days = everyone[settings.BENJI][person]["weight"], everyone.nodes[person]["days"]
    print(f"{person:14s} {near_days:5d} {all_days:4d}  {near_days / all_days:4.0%}")

In [ ]:
KIND_COLORS = {"centre": settings.BENJI_COLOR, "family": settings.FAMILY_COLOR, "name": settings.NAME_COLOR}
u, v, most = max(benji_star.edges(data="weight"), key=lambda e: e[2])
strongest = v if u == settings.BENJI else u
caption = (
    f"Zo lees je dit. In het midden Benji. Elke andere stip is iemand die we in de chat noemden: "
    f"grijs een naam, lichtblauw een familiewoord ('mon père' van Roos is vader Roos, van Louis "
    f"vader Louis). Hoe groter de stip, hoe meer dagen die persoon genoemd werd sinds de eerste "
    f"'Benji' ({benji_start:%d-%m-%Y}). Een lijn betekent: op die dag werd de persoon binnen "
    f"{settings.BENJI_WINDOW.seconds // 60} minuten van een bericht over Benji genoemd. Hoe dikker "
    f"de lijn, hoe meer dagen: de dikste, {strongest}, staat voor {most} dagen. Alleen wie op "
    f"minstens {settings.MIN_NEAR_DAYS} dagen bij Benji genoemd werd staat erin. Met de klok mee "
    f"van boven: van de sterkste naar de zwakste lijn. Een lijn zegt niet dat iemand heeft "
    f"opgepast, alleen dat de naam in de buurt viel."
)
benji_plot = PlotSettings(
    figsize=(11, 10),
    title=f"Wie noemen we in de buurt van Benji? (binnen {settings.BENJI_WINDOW.seconds // 60} "
          f"minuten, sinds {benji_start:%m-%Y})",
)
fig, ax = GraphPlot(benji_plot).plot(
    graph=benji_star,
    pos=star_layout(benji_star),
    colors={n: KIND_COLORS[d["country"]] for n, d in benji_star.nodes(data=True)},
    legend={"Benji": settings.BENJI_COLOR, "familie (familiewoord)": settings.FAMILY_COLOR,
            "naam": settings.NAME_COLOR},
    caption=caption,
    size_per_day=4.0,
)

## 7.4.11 Visualisations 7-10: who connects our friends and family?

All the people from `config.toml` `[people]` (names and family words, still a candidate list),
over the **whole chat**. Benji is not a node: he had his own picture. My rules (2026-11-03,
`settings.FRIEND_WINDOW`, `MIN_PERSON_DAYS`, `MIN_PAIR_DAYS`, `N_CONNECTORS`):

1. **Two edge rules, side by side**, because a centrality score is a claim about the rule
   (07.2): *two people are linked on a day when their names came up within 30 minutes of
   each other* (07.1's "nearby"), and the strict one: *in the same message*.
2. **Whose side** is my rule from visualisation 5: Roos names them on 60%+ of their days is
   Roos's side, Louis 60%+ is Louis's, in between is shared. "Naming someone more" is not the
   same as "my friend".
3. **The connector is the highest betweenness** (07.2: the share of shortest paths between
   two others that run through you), unweighted, computed on **all** people and links. Only
   the picture is pruned (10+ days, links of 3+ days, top 3 per person).
4. **The matrix** shows the same graph without pruning: one row and one column per person
   named on 10+ days, grouped by side.
5. **The same-message picture is different** (Roos, 2026-11-03): only **since Benji's birth**,
   **with Benji as a node**, people named on fewer than `MIN_PERSON_DAYS` days left out (all
   links between the rest stay), and **no circle** but a spring layout (Kamada-Kawai): people
   named together are pulled together, so a position means "belongs with", not a side. The
   table and the matrices above it still cover the whole chat, without Benji.

In [ ]:
friends_named = mentions(messages, people)
sides = person_sides(friends_named)
print("sides:", sides.value_counts().to_dict(), "\n")

rules = {
    f"binnen {settings.FRIEND_WINDOW.seconds // 60} minuten": people_graph(friends_named, settings.FRIEND_WINDOW),
    "in hetzelfde bericht": people_graph(friends_named),
}
connectors = {}
for rule, graph in rules.items():
    nx.set_node_attributes(graph, sides.to_dict(), "side")
    between = pd.Series(nx.betweenness_centrality(graph)).sort_values(ascending=False)
    connectors[rule] = tuple(between.index[: settings.N_CONNECTORS])
    print(f"{rule}: {graph.number_of_nodes()} people, {graph.number_of_edges()} links, "
          f"density {nx.density(graph):.2f}, {nx.number_connected_components(graph)} piece(s)")
    for person in between.index[:8]:
        print(f"  {person:14s} betweenness {between[person]:.2f}  degree {graph.degree(person):2d}  "
              f"{graph.nodes[person]['days']:3d} days  {sides[person]}")
    print()
both = set.intersection(*(set(c) for c in connectors.values()))
print("in the top", settings.N_CONNECTORS, "under both rules:", ", ".join(sorted(both)) or "-")

In [ ]:
SIDE_COLORS = {"Roos": settings.ROOS_COLOR, "Louis": settings.LOUIS_COLOR, "shared": settings.SHARED_COLOR}
low, high = settings.SHARED_LOW, settings.SHARED_HIGH
side_legend = {
    f"Roos noemt ze vaker ({high:.0%}+)": settings.ROOS_COLOR,
    f"gedeeld ({low:.0%} - {high:.0%})": settings.SHARED_COLOR,
    f"Louis noemt ze vaker ({1 - low:.0%}+)": settings.LOUIS_COLOR,
}


def friends_caption(shown, rule):
    u, v, most = max(shown.edges(data="weight"), key=lambda e: e[2])
    return (
        f"Zo lees je dit. Elke stip is iemand uit onze chat, gekleurd naar wie die persoon vaker "
        f"noemt (in dagen): rechts de kant van Roos, onderaan gedeeld, links de kant van Louis. "
        f"Hoe groter de stip, hoe meer dagen genoemd. Een lijn: de twee namen vielen {rule}; hoe "
        f"dikker, hoe meer dagen (de dikste, {u}-{v}, {most} dagen). Een ring: de "
        f"{settings.N_CONNECTORS} grootste verbinders, de mensen die het vaakst op de kortste "
        f"route tussen twee anderen liggen (betweenness), berekend op alle mensen en lijnen. In "
        f"beeld alleen wie op minstens {settings.MIN_PERSON_DAYS} dagen genoemd werd, lijnen van "
        f"minstens {settings.MIN_PAIR_DAYS} dagen, per persoon de {settings.MAX_LINKS} sterkste."
    )


rule = f"binnen {settings.FRIEND_WINDOW.seconds // 60} minuten"  # same message: next cell
graph = rules[rule]
shown = prune(graph, min_days=settings.MIN_PERSON_DAYS, min_link=settings.MIN_PAIR_DAYS, keep=())
friends_plot = PlotSettings(figsize=(12, 11), title=f"Wie verbindt onze vrienden en familie? Namen {rule}")
fig, ax = GraphPlot(friends_plot).plot(
    graph=shown,
    pos=circle_layout(shown, countries=("Roos", "shared", "Louis"), attr="side"),
    colors={n: SIDE_COLORS[d["side"]] for n, d in shown.nodes(data=True)},
    legend=side_legend,
    caption=friends_caption(shown, rule),
    size_per_day=3.0,
    marked=connectors[rule],
    marked_label=f"top {settings.N_CONNECTORS} verbinders (betweenness)",
)

In [ ]:
people_and_benji = {**people, "names": {**people["names"], settings.BENJI: settings.BENJI_PATTERN}}
since_birth = messages[messages["ts_local"] >= settings.BIRTH_DATE]
birth_named = mentions(since_birth, people_and_benji)
birth_sides = person_sides(birth_named)
birth_graph = people_graph(birth_named)  # same message
birth_between = pd.Series(nx.betweenness_centrality(birth_graph)).sort_values(ascending=False)
birth_connectors = tuple(birth_between.index[: settings.N_CONNECTORS])
print(f"since {settings.BIRTH_DATE:%d %b %Y}, same message: {birth_graph.number_of_nodes()} people "
      f"(Benji included), {birth_graph.number_of_edges()} links, "
      f"{nx.number_connected_components(birth_graph)} piece(s)")
for person in birth_between.index[:8]:
    print(f"  {person:14s} betweenness {birth_between[person]:.2f}  degree {birth_graph.degree(person):2d}  "
          f"{birth_graph.nodes[person]['days']:3d} days  {birth_sides[person]}")

birth_shown = nx.Graph(birth_graph.subgraph(
    n for n, d in birth_graph.nodes(data=True)
    if d["days"] >= settings.MIN_PERSON_DAYS or n == settings.BENJI
))
print(f"\nin the picture: {birth_shown.number_of_nodes()} people, {birth_shown.number_of_edges()} links; "
      f"left out: {birth_graph.number_of_nodes() - birth_shown.number_of_nodes()} people named on "
      f"fewer than {settings.MIN_PERSON_DAYS} days")

u, v, most = max(birth_shown.edges(data="weight"), key=lambda e: e[2])
birth_caption = (
    f"Zo lees je dit. Elke stip is iemand uit onze chat sinds de geboorte van Benji "
    f"({settings.BIRTH_DATE:%d-%m-%Y}); oranje is Benji, de andere kleuren zeggen wie die persoon "
    f"vaker noemt (in dagen). Hoe groter de stip, hoe meer dagen genoemd. Een lijn: de twee namen "
    f"stonden in hetzelfde bericht; hoe dikker, hoe meer dagen (de dikste, {u}-{v}, {most} dagen). "
    f"Mensen die vaak samen genoemd worden staan dicht bij elkaar; verder zegt de plek niets. Een "
    f"ring: de {settings.N_CONNECTORS} grootste verbinders, de mensen die het vaakst op de kortste "
    f"route tussen twee anderen liggen (betweenness), berekend op iedereen sinds de geboorte. In "
    f"beeld alleen wie op minstens {settings.MIN_PERSON_DAYS} dagen genoemd werd, met alle lijnen."
)
# Kamada-Kawai: people linked by a short path are placed close (unweighted, like the
# betweenness). Stretched to fill the square: only "close or far" means something anyway.
layout = nx.rescale_layout_dict(nx.kamada_kawai_layout(birth_shown, weight=None), scale=1.0)
tallest = max(abs(y) for _, y in layout.values())
birth_pos = {n: (x, y / tallest) for n, (x, y) in layout.items()}

birth_plot = PlotSettings(
    figsize=(12, 10),
    title="Wie verbindt onze vrienden en familie? Namen in hetzelfde bericht, sinds de geboorte van Benji",
)
fig, ax = GraphPlot(birth_plot).plot(
    graph=birth_shown,
    pos=birth_pos,
    colors={n: settings.BENJI_COLOR if n == settings.BENJI else SIDE_COLORS[birth_sides[n]]
            for n in birth_shown},
    legend={"Benji": settings.BENJI_COLOR, **side_legend},
    caption=birth_caption,
    size_per_day=2.0,
    marked=birth_connectors,
    marked_label=f"top {settings.N_CONNECTORS} verbinders (betweenness)",
)

In [ ]:
for rule, graph in rules.items():
    named_enough = graph.subgraph(n for n, d in graph.nodes(data=True) if d["days"] >= settings.MIN_PERSON_DAYS)
    order = side_order(named_enough, sides)
    groups = [sum(sides[n] == side for n in order) for side in ("Roos", "shared", "Louis")]
    matrix_plot = PlotSettings(figsize=(12, 12), title=f"Dezelfde mensen als matrix: namen {rule}")
    fig, ax = MatrixPlot(matrix_plot).plot(
        graph=named_enough,
        order=order,
        label_colors={n: SIDE_COLORS[sides[n]] for n in order},
        groups=groups,
        marked=connectors[rule],
        caption=(
            f"Zo lees je dit. Elke rij en kolom is iemand die op minstens {settings.MIN_PERSON_DAYS} "
            f"dagen genoemd werd, gegroepeerd: eerst de kant van Roos (paarse namen), dan gedeeld "
            f"(grijs), dan de kant van Louis (groen). Een vakje kleurt als de twee namen {rule} "
            f"vielen: hoe donkerder, hoe meer dagen; wit is nooit. De blokken op de diagonaal zijn "
            f"mensen van dezelfde kant, de blokken ernaast verbinden de kanten. Vetgedrukt: de "
            f"{settings.N_CONNECTORS} grootste verbinders (betweenness)."
        ),
    )

## 7.4.12 Visualisation 11: what do we talk about?

The 15 most discussed topics, hobbies and activities, and which of them come up together.
My rules (2026-11-03, `settings.ACTIVITY_TOPICS`, `N_TOPICS`):

1. **Topics are word lists** in Dutch, French and English, a *candidate* list like the people:
   27 topics, checked on samples. Left out because they mostly meant something else: "dans"
   (French "in"), "les", "course" ("of course"), "soirée", "photo" (sending one), "gym" (the
   climbing gym), "boeken" (booking). Links are removed first: a Spotify URL is no music.
2. **Not only hobbies**: Benji, work, study and holidays count too (my choice).
3. **The 15 most discussed**, counted in days, like the places.
4. **The edge rule, as a sentence:** *two topics are linked on a day when they came up in the
   same message.* I first tried "the same day", as with the places: then every topic was linked
   to every other one (105 of 105 links), and the thickest lines were simply the biggest topics.
5. **Clutter and layout** as before: links of 5+ days, the 3 strongest per topic (7.4.5), a
   spring layout (Kamada-Kawai). The colour is who names a topic more often (60/40).

In [ ]:
import warnings

# TOPIC_PATTERNS from lesson 4 have capture groups; str.contains only needs a yes/no
warnings.filterwarnings("ignore", "This pattern is interpreted as a regular expression")

unlinked = messages.assign(message=messages["message"].str.replace(settings.LINK_PATTERN, " ", regex=True))
topic_days = place_days(unlinked, places=settings.ACTIVITY_TOPICS)
ranking = topic_days.groupby(["place", "country"]).size().rename("days").reset_index()
ranking = ranking.sort_values("days", ascending=False).reset_index(drop=True)
top_topics = ranking["place"].head(settings.N_TOPICS).tolist()
for i, row in ranking.iterrows():
    mark = "" if i < settings.N_TOPICS else "   (not in the top)"
    print(f"{i + 1:2d}. {row['place']:20s} {row['country']:12s} {row['days']:4d} days{mark}")

In [ ]:
# Same message (Roos, 2026-11-03). The people functions do the counting: a topic is
# matched like a name in "names", whoever writes it.
patterns = {topic: settings.ACTIVITY_TOPICS[topic][1] for topic in top_topics}
topic_mentions = mentions(unlinked, {"names": patterns})
topics = people_graph(topic_mentions)  # no window: same message; weight in days
topic_sides = person_sides(topic_mentions)
heaviest = sorted(topics.edges(data="weight"), key=lambda e: -e[2])
print(f"{topics.number_of_nodes()} topics, {topics.number_of_edges()} links of "
      f"{settings.N_TOPICS * (settings.N_TOPICS - 1) // 2} possible, density {nx.density(topics):.2f}")
print("strongest:", ", ".join(f"{u}-{v} {w}" for u, v, w in heaviest[:6]))

topics_shown = prune(topics, min_days=0, keep=())
print(f"in the picture: {topics_shown.number_of_edges()} links "
      f"(at least {settings.MIN_LINK_DAYS} days, top {settings.MAX_LINKS} per topic)")
layout = nx.rescale_layout_dict(nx.kamada_kawai_layout(topics_shown, weight=None), scale=1.0)
tallest = max(abs(y) for _, y in layout.values())
topic_pos = {n: (x, y / tallest) for n, (x, y) in layout.items()}

u, v, most = heaviest[0]
topic_caption = (
    f"Zo lees je dit. Elke stip is een van onze {settings.N_TOPICS} meest besproken onderwerpen, "
    f"gevonden met zoekwoorden in het Nederlands, Frans en Engels; hoe groter de stip, hoe meer "
    f"dagen het onderwerp in de chat voorkwam. De kleur zegt wie het vaker noemt (in dagen). Een "
    f"lijn: de twee onderwerpen stonden in hetzelfde bericht; hoe dikker, hoe meer dagen "
    f"(de dikste, {u}-{v}, {most} dagen). Alleen lijnen van minstens {settings.MIN_LINK_DAYS} "
    f"dagen, per onderwerp de {settings.MAX_LINKS} sterkste. Onderwerpen die vaak samen voorkomen "
    f"staan dicht bij elkaar; verder zegt de plek niets."
)
topic_plot = PlotSettings(figsize=(12, 10), title="Waar praten we over? Onze 15 meest besproken onderwerpen")
fig, ax = GraphPlot(topic_plot).plot(
    graph=topics_shown,
    pos=topic_pos,
    colors={n: SIDE_COLORS[topic_sides[n]] for n in topics_shown},
    legend=side_legend,
    caption=topic_caption,
    size_per_day=1.5,
)

## 7.4.13 Visualisation 12: van onderweg naar thuis

What do we talk about when we name a place? Five cities, six themes in two groups. My rules
(2026-11-12, `settings.HOME_PLACES`, `HOME_THEMES`, `HOME_WINDOW`, `HOME_MIN_SHARE`):

1. **Places**: Parijs (Gare du Nord included), Amsterdam, Utrecht, Zoetermeer, Nieuwegein,
   with the regexes of `settings.PLACES`.
2. **Themes** as word lists (NL/FR/EN), a *candidate* list like the topics in 7.4.12:
   *onderweg* = reizen, klimmen & sport, eten & uitgaan; *thuis* = Benji & kind, wonen,
   familie. Work and holidays are left out: they hardly differ between the places.
3. **The link rule, as a sentence:** *a place and a theme belong together in a "gesprekje":
   the message naming the place and the 3 messages after it*, whoever writes them.
4. **share(place, theme)** = gesprekjes with the theme / messages with the place.
   **thuis-score** = the summed home shares / the summed shares of all six themes.
5. **Picture** (after the shuffle test below, whose result goes in the text under it): a
   place sits at x = its thuis-score; lines below 8% are left out; only the
   two extremes, Parijs and Nieuwegein, get their percentages. No time axis.

Known loose ends: "Gare du Nord" is part of Parijs *and* matches `gare` in reizen; sport
reuses the lesson-3 lists ("km", the climbing grades); "maison" is also "à la maison".

In [ ]:
from roos_pipeline.home import HomeAwayPlot, theme_shares

home_table = theme_shares(messages)
print(f"{'':12s} {'thuis':>6s} {'msgs':>5s}  " + "  ".join(f"{t:>15s}" for t in settings.HOME_THEMES))
for place, row in home_table.sort_values("thuis_score").iterrows():
    print(f"{place:12s} {row['thuis_score']:6.0%} {int(row['berichten']):5d}  "
          + "  ".join(f"{row[t]:15.1%}" for t in settings.HOME_THEMES))

### Is Nieuwegein really more "thuis met kind"? A shuffle test

The picture puts Nieuwegein far to the right of Parijs, Amsterdam and Utrecht, but on 60
messages. Could a gap like that come from chance alone? The check from 04.6
(`NullDistribution`):

1. **The number**: thuis-score of Nieuwegein minus the thuis-score of Parijs, Amsterdam and
   Utrecht taken together (`home_gap`). My claim is "greater", so the test is one-sided.
2. **Shuffle the place label** over all messages naming exactly one of the five places
   (`place_talks`; a message naming two places has no single label and is left out), 10,000
   times. In that world a gesprekje's themes have nothing to do with the place it names.
3. **Second check, a day as the unit**: messages on the same day are no independent draws
   (one evening about a house is ten messages). So the same test once more on one row per
   (day, place), with the themes of that day summed.

What a small p does *not* say: that the themes were found correctly (the word lists), or
*why* Nieuwegein differs. "Sinds de komst van Benji" is a claim about time that this
picture, without a time axis, does not test.

In [ ]:
from goad_toolkit.analytics import NullDistribution
from goad_toolkit.visualizer import NullPlot

from roos_pipeline.home import home_gap, place_talks

talks = place_talks(messages)
talk_days = talks.groupby(["day", "place"])[["home", "away"]].sum().reset_index()
units = {"messages": talks, "days": talk_days}
shuffle = {}  # unit -> NullResult, for the caption under the picture
for unit, frame in units.items():
    shuffle[unit] = null = NullDistribution(home_gap, n_iter=settings.N_SHUFFLE, seed=settings.SEED).run(frame, label="place")
    reached = int((null.null >= null.observed).sum())
    print(f"unit = {unit:8s} Nieuwegein {sum(frame['place'] == 'Nieuwegein'):3d}, "
          f"Parijs/Amsterdam/Utrecht {frame['place'].isin(['Parijs', 'Amsterdam', 'Utrecht']).sum():4d}  "
          f"gap {null.observed:+.0%}  shuffles reaching it: {reached} of {null.n_iter:,}  "
          f"p = {null.p_value('greater'):.4f}")

null_plot = PlotSettings(
    figsize=(10, 4),
    title="Nieuwegein vs. Parijs, Amsterdam en Utrecht: verschil in thuis-score, per dag",
    xlabel="verschil in thuis-score als de plek willekeurig wordt gehusseld",
    ylabel="dichtheid",
)
fig, ax = NullPlot(null_plot).plot(result=shuffle["days"], alternative="greater", label="echte verschil")

In [ ]:
# The picture, with the shuffle test of the cell above in the text underneath.
by_message, by_day = shuffle["messages"], shuffle["days"]


def reaching(null):
    return int((null.null >= null.observed).sum())


n_shuffles = f"{by_message.n_iter:,}".replace(",", ".")  # Dutch thousands: 10.000
p_floor = f"{1 / (by_message.n_iter + 1):.4f}".replace(".", ",")  # the smallest p a shuffle can give


home_caption = (
    f"Elke donkere stip is een stad. Hoe verder naar rechts, hoe hoger de thuis-score: "
    f"het deel van de thema's in de gesprekken over die stad dat over thuis met kind gaat, tegenover "
    f"spontaan op pad. Een lijn verbindt een stad met een thema: hoe dikker, hoe vaker dat thema voorkwam "
    f"in een gesprekje over de stad (het bericht dat de stad noemt plus de {settings.HOME_WINDOW} "
    f"berichten erna). Lijnen onder {settings.HOME_MIN_SHARE:.0%} zijn weggelaten.\n\n"
    f"De zes thema's vallen in twee groepen: spontaan op pad (reizen, klimmen & sport, eten & uitgaan) en "
    f"thuis met kind (Benji & kind, wonen, familie). Werk en vakantie tellen niet mee. Een thema is "
    f"gevonden met een woordenlijst in het Nederlands, Frans en Engels, bijvoorbeeld trein, TGV en gare; "
    f"klimmen, escalade en hardlopen; restaurant en apéro; Benji, crèche en luiers; huis, appartement en "
    f"verhuizen; ouders, oma en mes parents. Een woord kan ook iets anders betekenen.\n\n"
    f"Schudtest. De thuis-score van Nieuwegein ligt {by_message.observed * 100:.0f} procentpunt hoger dan die van Parijs, "
    f"Amsterdam en Utrecht samen. We hebben {n_shuffles} keer willekeurig gehusseld welke stad "
    f"bij welk gesprekje hoort. In {reaching(by_message)} van de {n_shuffles} keer kwam er een "
    f"even groot verschil uit (p < {p_floor}); met een dag als eenheid in plaats "
    f"van een bericht {reaching(by_day)} keer. Het verschil is dus vrijwel zeker geen toeval; waarom het er is, "
    f"zegt de test niet."
)

home_plot = PlotSettings(
    figsize=(16, 10.5),
    title="De reis van onze relatie: van spontaan op pad naar samen thuis.",
)
fig, ax = HomeAwayPlot(home_plot).plot(
    table=home_table,
    subtitle=(
        "Tussen Parijs, Amsterdam en Utrecht ging het over reizen, sport en uitgaan. Sinds de komst van Benji\n"
        "gaat het in Nieuwegein over wonen en ons kind, met hulp van opa en oma in Zoetermeer."
    ),
    caption=home_caption,
)
fig.savefig(Paths().figures / "onderweg_naar_thuis.png", dpi=170, facecolor=settings.PAPER, bbox_inches="tight")

## 7.4.14 What to write down, in my own words

The titles above only say *what* is drawn. The finding is for me to write.

1. **My edge rule, as a sentence**, including the window (a day). What can it *not* tell
   apart? (Think of one day where Paris and Barcelona came up together for reasons that have
   nothing to do with each other.)
2. **One number that is not a picture**: the density, the share of links resting on a single
   day, or the place-days per month.
3. **What I did about the nodes**: the hand-made place list, what I left out, Corsica kept
   against the rules. Would Corte and Piana belong *in* Corsica instead of next to it?
4. **The x-axis**: a dot sits at its median day. What does that hide for Paris, which came up
   all through the period? Would the *first* day suit the story I want to tell better?
5. **Rules 3 and 4**: why top 3 of *one* place, and why nobody alone? What do the links of
   Corsica and Terschelling (below 5 days) say, and what don't they?
6. **The title.** Does "the cities stop during the pregnancy" survive 7.4.6? If not, what is
   the sentence that does?
7. **The story of visualisation 2** (feedback, 2026-11-03): what is the one thing a reader
   should see? Which lines carry it, and what are the other places: holidays, family, work?
8. **Visualisation 5**: is 60% a lot, next to Roos's share of all place-days? Which places sit
   close to a boundary, and would they change colour with 55/45?
9. **Visualisation 6**: is "named near Benji" the same as "could babysit"? Which names are
   near Benji for another reason (a birth announcement, a present, a visit)? And who is
   *missing* because we call them something the list doesn't know?
10. **Visualisations 7-10**: the two rules give different connectors. Which rule fits the
    sentence "X connects our worlds", and what does it mean that one person is in both top 3s?
    Betweenness ignores the weights: a link of 1 day counts as much as one of 27. Fair?
11. **Visualisation 11**: the biggest topics still have the thickest lines (werk, klimmen,
    hardlopen are in many messages). Which thin line surprises you, and does it hold up when you
    read five of its messages? And what is *missing* because its words are not on the list?
12. **Visualisation 12**: the thuis-score of Nieuwegein rests on 60 messages, Parijs on 518.
    How much would one week of house-hunting move Nieuwegein? Read five gesprekjes per line
    you want to claim (Zoetermeer-familie, Nieuwegein-wonen): is it what the theme says?
13. **The subtitle** makes three claims: "sinds de komst van Benji" (time), "wonen en ons
    kind" (Nieuwegein) and "met hulp van opa en oma" (Zoetermeer). Which does the picture
    show, which does the shuffle test, and which needs another check? Zoetermeer has familie
    27%, but also reizen 24%: is "family" the same as "help"?
